<a href="https://colab.research.google.com/github/mpw2004/UoM-AI-session-term-1/blob/main/Untitled2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##importing needed libraries

In [22]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score

##uploading the data file

In [23]:
uploaded = files.upload()
df = pd.read_csv('housing_price_prediction.csv')


Saving housing_price_prediction.csv to housing_price_prediction (2).csv


## Display initial information

In [24]:
print("\n--- Data Inspection (First 5 Rows) ---")
print(df.head())
print("\n--- Data Information ---")
df.info()


--- Data Inspection (First 5 Rows) ---
      price  area  bedrooms  bathrooms  stories mainroad guestroom basement  \
0  13300000  7420         4          2        3      yes        no       no   
1  12250000  8960         4          4        4      yes        no       no   
2  12250000  9960         3          2        2      yes        no      yes   
3  12215000  7500         4          2        2      yes        no      yes   
4  11410000  7420         4          1        2      yes       yes      yes   

  hotwaterheating airconditioning  parking prefarea furnishingstatus  
0              no             yes        2      yes        furnished  
1              no             yes        3       no        furnished  
2              no              no        2      yes   semi-furnished  
3              no             yes        3      yes        furnished  
4              no             yes        2       no        furnished  

--- Data Information ---
<class 'pandas.core.frame.DataFra

##Converting the given details to numeric values

In [25]:
# 1.1 Convert 'Yes'/'No' columns to 1/0
binary_cols = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']

# Create a mapping function
def map_yes_no(value):
    return 1 if value == 'yes' else 0

# Apply the mapping to the specified columns
df[binary_cols] = df[binary_cols].apply(lambda x: x.map(map_yes_no))

print("\n--- Data after mapping 'yes'/'no' to 1/0 ---")
print(df[binary_cols].head())



--- Data after mapping 'yes'/'no' to 1/0 ---
   mainroad  guestroom  basement  hotwaterheating  airconditioning  prefarea
0         1          0         0                0                1         1
1         1          0         0                0                1         0
2         1          0         1                0                0         1
3         1          0         1                0                1         1
4         1          1         1                0                1         0


##Handle the 'furnishingstatus' (Multi-class categorical)

In [26]:
# We use One-Hot Encoding (pd.get_dummies) to create new binary columns
df = pd.get_dummies(df, columns=['furnishingstatus'], drop_first=True)
# drop_first=True avoids multicollinearity

print("\n--- Data after One-Hot Encoding for Furnishing Status ---")
print(df.filter(regex='furnishing').head())



--- Data after One-Hot Encoding for Furnishing Status ---
   furnishingstatus_semi-furnished  furnishingstatus_unfurnished
0                            False                         False
1                            False                         False
2                             True                         False
3                            False                         False
4                            False                         False


##Feature and Target Separation

In [27]:
# Separate the target variable (price) from the features (X)
X = df.drop('price', axis=1)
y = df['price']

print(f"\nTotal Features: {X.shape[1]}")


Total Features: 13


##Scaling Numerical Features

In [28]:
# Select all numerical columns that are NOT the binary-encoded features
# We will scale 'area', 'bedrooms', 'bathrooms', 'stories', 'parking'
scaler = MinMaxScaler()
numerical_cols = ['area', 'bedrooms', 'bathrooms', 'stories', 'parking']
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

print("\n--- Scaled Features (First 5 Rows) ---")
print(X[numerical_cols].head())


--- Scaled Features (First 5 Rows) ---
       area  bedrooms  bathrooms   stories   parking
0  0.396564       0.6   0.333333  0.666667  0.666667
1  0.502405       0.6   1.000000  1.000000  1.000000
2  0.571134       0.4   0.333333  0.333333  0.666667
3  0.402062       0.6   0.333333  0.333333  1.000000
4  0.396564       0.6   0.000000  0.333333  0.666667


##Split Data into Training and Testing Sets

In [29]:
# Split the data with a 70/30 ratio for training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"\nTraining set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")


Training set size: 381
Testing set size: 164


## Model Training and Prediction

In [30]:
# Initialize the Linear Regression Model
model = LinearRegression()

# Train the model
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

##Model Evaluation

In [31]:
# Calculate the R-squared (R2) score, which measures how well the model fits the data
r2 = r2_score(y_test, y_pred)

print("\n==============================================")
print("             MODEL EVALUATION")
print("==============================================")
print(f"R-squared Score (Test Set): {r2:.4f}")
print("---")
print("R-squared represents the proportion of the variance for a dependent variable")
print("that's explained by the independent variables in a regression model.")
print("(A score closer to 1.0 is better.)")
print("==============================================")



             MODEL EVALUATION
R-squared Score (Test Set): 0.6463
---
R-squared represents the proportion of the variance for a dependent variable
that's explained by the independent variables in a regression model.
(A score closer to 1.0 is better.)


## This shows the 'importance' and direction of each feature

In [32]:
print("\n--- Model Coefficients (Feature Importance) ---")
coefficients = pd.Series(model.coef_, index=X.columns)
print(coefficients.sort_values(ascending=False))


--- Model Coefficients (Feature Importance) ---
area                               3.685330e+06
bathrooms                          3.344254e+06
stories                            1.251803e+06
parking                            9.093337e+05
airconditioning                    6.858393e+05
hotwaterheating                    6.163754e+05
prefarea                           5.091921e+05
basement                           4.826035e+05
mainroad                           4.080737e+05
bedrooms                           4.044657e+05
guestroom                          2.757105e+05
furnishingstatus_semi-furnished   -1.216527e+05
furnishingstatus_unfurnished      -3.911912e+05
dtype: float64
